# Cosmos DB connectivity smoke test

This notebook validates end-to-end access to the `customers` container in Azure Cosmos DB using Databricks secrets.

- Check the available secret scopes and keys
- Build the Cosmos DB connection configuration
- Verify read access from the container
- Write a sample customer record
- Confirm the write and query Gold-tier customers


In [0]:
print([s.name for s in dbutils.secrets.listScopes()])
print([s.key for s in dbutils.secrets.list("kv-poc")])

In [0]:
cosmos_cfg = {
    "spark.cosmos.accountEndpoint": "https://cosmos-dbx-poc-sm.documents.azure.com:443/",
    "spark.cosmos.accountKey": dbutils.secrets.get("kv-poc", "cosmos-a-key"),
    "spark.cosmos.database": "pocdb",
    "spark.cosmos.container": "customers",
    "spark.cosmos.read.inferSchema.enabled": "true",
}
print(cosmos_cfg["spark.cosmos.accountKey"])

In [0]:
df = spark.read.format("cosmos.oltp").options(**cosmos_cfg).load()
display(df.select("id", "customerId", "name", "city", "tier", "tenant"))

In [0]:
from pyspark.sql import Row

new_df = spark.createDataFrame([
    Row(id="6", customerId="C006", name="Vikram Joshi", city="Chandigarh", tier="Gold", tenant="A")
])

(new_df.write.format("cosmos.oltp")
    .options(**cosmos_cfg)
    .option("spark.cosmos.write.strategy", "ItemOverwrite")
    .mode("append")
    .save())

In [0]:
print("Total rows:", spark.read.format("cosmos.oltp").options(**cosmos_cfg).load().count())

gold_df = (spark.read.format("cosmos.oltp")
    .options(**cosmos_cfg)
    .option("spark.cosmos.read.customQuery", "SELECT * FROM c WHERE c.tier = 'Gold'")
    .load())
display(gold_df.select("id", "name", "tier"))